# ChurnInsight — Logistic Regression + SMOTE (Contrato v4) — FINAL

> **Objetivo:** versión **post-MVP** (roadmap) con **Regresión Logística + SMOTE**, alineada al **Contrato v4**, lista para **Run All** en Google Colab/Jupyter.

- Modelo: **Logistic Regression** (interpretable) + **SMOTE** (balanceo)
- Salida: `forecast`, `probability`, `top_features` (explicabilidad)
- Artefactos: `joblib` + `metadata.json` + `CSV demo para Power BI`

⚠️ Este notebook **no reemplaza** el MVP v3. Es una **versión de roadmap**, separada y compatible con backend vía contrato v4.

## 1) Setup (imports + configuración)

Este bloque instala dependencias si faltan (Colab), define parámetros (seed/threshold) y rutas de salida.

In [ ]:
import sys, os, json, warnings, datetime
warnings.filterwarnings("ignore")

# En Colab, a veces falta imbalanced-learn
try:
    import imblearn  # noqa
except Exception:
    !pip -q install imbalanced-learn

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

SEED = 42
np.random.seed(SEED)

# Umbral de negocio (alineado con backend)
THRESHOLD = 0.22

ARTIFACT_DIR = "models"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

MODEL_FILENAME = "churn_logreg_smote_v4.joblib"
METADATA_FILENAME = "metadata_logreg_smote_v4.json"
POWERBI_CSV_FILENAME = "predictions_powerbi_demo_v4.csv"

MODEL_PATH = os.path.join(ARTIFACT_DIR, MODEL_FILENAME)
METADATA_PATH = os.path.join(ARTIFACT_DIR, METADATA_FILENAME)
POWERBI_CSV_PATH = os.path.join(ARTIFACT_DIR, POWERBI_CSV_FILENAME)

print("✅ Setup listo")
print("Model path:", MODEL_PATH)
print("Metadata path:", METADATA_PATH)
print("PowerBI CSV path:", POWERBI_CSV_PATH)


## 2) Dataset (real si existe, si no: demo sintético)

El notebook intenta cargar un dataset típico (`Churn_Modelling.csv`). Si no lo encuentra, genera un dataset demo con las columnas del contrato v4 + objetivo `Exited`.

In [ ]:
CANDIDATE_PATHS = [
    "Churn_Modelling.csv",
    "data/Churn_Modelling.csv",
    "data/churn.csv",
    "churn.csv",
]

df = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"✅ Dataset cargado desde: {p} | shape={df.shape}")
        break

if df is None:
    n = 10000
    geos = np.random.choice(["Spain","France","Germany"], size=n, p=[0.45,0.35,0.20])
    gender = np.random.choice(["Male","Female"], size=n, p=[0.55,0.45])
    age = np.random.randint(18, 101, size=n)
    credit = np.random.randint(100, 1001, size=n)
    balance = np.round(np.random.gamma(shape=2.0, scale=1500.0, size=n), 2)
    salary = np.round(np.random.gamma(shape=2.0, scale=20000.0, size=n), 2)
    tenure = np.random.randint(0, 21, size=n)
    numprod = np.random.choice([1,2,3,4], size=n, p=[0.55,0.30,0.10,0.05])
    sat = np.random.choice([1,2,3,4,5], size=n, p=[0.10,0.20,0.35,0.25,0.10])
    is_active = np.random.choice([0,1], size=n, p=[0.40,0.60])
    has_card = np.random.choice([0,1], size=n, p=[0.35,0.65])
    complain = np.random.choice([0,1], size=n, p=[0.80,0.20])

    logit = (
        -2.0
        + 0.02*(age-40)
        + 0.001*(balance/100)
        + 0.8*complain
        - 0.6*is_active
        + 0.15*(numprod-1)
        + 0.002*((1000-credit)/10)
    )
    p_true = 1/(1+np.exp(-logit))
    exited = (np.random.rand(n) < p_true).astype(int)

    df = pd.DataFrame({
        "Geography": geos,
        "Gender": gender,
        "Age": age,
        "CreditScore": credit,
        "Balance": balance,
        "EstimatedSalary": salary,
        "Tenure": tenure,
        "NumOfProducts": numprod,
        "SatisfactionScore": sat,
        "IsActiveMember": is_active,
        "HasCrCard": has_card,
        "Complain": complain,
        "Exited": exited
    })
    print(f"✅ Dataset DEMO generado | shape={df.shape}")

if "Exited" not in df.columns:
    if "Churn" in df.columns:
        df = df.rename(columns={"Churn":"Exited"})
    elif "target" in df.columns:
        df = df.rename(columns={"target":"Exited"})

assert "Exited" in df.columns, "No se encontró columna target ('Exited')."
df.head()


## 3) Contrato v4 (Backend camelCase → DS PascalCase) + validación

Incluye funciones:
- `contract_v4_to_df(payload)`
- `validate_domain_v4(payload)`

In [ ]:
CONTRACT_V4_FIELDS = [
    "geography", "gender", "age", "creditScore", "balance", "estimatedSalary",
    "tenure", "numOfProducts", "satisfactionScore", "isActiveMember",
    "hasCrCard", "complain"
]

V4_REQUIRED = {"geography","age","creditScore","balance","numOfProducts","satisfactionScore","isActiveMember","complain"}
V4_OPTIONAL = set(CONTRACT_V4_FIELDS) - V4_REQUIRED

V4_TO_DS = {
    "geography": "Geography",
    "gender": "Gender",
    "age": "Age",
    "creditScore": "CreditScore",
    "balance": "Balance",
    "estimatedSalary": "EstimatedSalary",
    "tenure": "Tenure",
    "numOfProducts": "NumOfProducts",
    "satisfactionScore": "SatisfactionScore",
    "isActiveMember": "IsActiveMember",
    "hasCrCard": "HasCrCard",
    "complain": "Complain",
}

def _to_bool01(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, (int, np.integer)):
        return 1 if int(x) != 0 else 0
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true","t","yes","y","1"}: return 1
        if s in {"false","f","no","n","0"}: return 0
    try:
        return 1 if float(x) != 0 else 0
    except Exception:
        return np.nan

def contract_v4_to_df(payload: dict) -> pd.DataFrame:
    row = {k: payload.get(k, np.nan) for k in CONTRACT_V4_FIELDS}
    for b in ["isActiveMember","hasCrCard","complain"]:
        row[b] = _to_bool01(row.get(b, np.nan))
    mapped = {V4_TO_DS[k]: row[k] for k in CONTRACT_V4_FIELDS}
    return pd.DataFrame([mapped])

def validate_domain_v4(payload: dict):
    w = []
    missing = [k for k in V4_REQUIRED if k not in payload or payload.get(k, None) is None]
    if missing:
        w.append(f"Faltan campos obligatorios: {missing}")

    geo = payload.get("geography", None)
    if geo is not None and str(geo) not in {"Spain","France","Germany"}:
        w.append("geography fuera de dominio esperado: Spain/France/Germany")

    gen = payload.get("gender", None)
    if gen is not None and str(gen) not in {"Male","Female"}:
        w.append("gender fuera de dominio esperado: Male/Female")

    def _check_int(name, lo, hi):
        v = payload.get(name, None)
        if v is None: return
        try:
            vi = int(v)
            if vi < lo or vi > hi:
                w.append(f"{name} fuera de rango [{lo},{hi}]")
        except Exception:
            w.append(f"{name} no es int válido")

    def _check_float(name, lo=0.0):
        v = payload.get(name, None)
        if v is None: return
        try:
            vf = float(v)
            if vf < lo:
                w.append(f"{name} debe ser ≥ {lo}")
        except Exception:
            w.append(f"{name} no es float válido")

    _check_int("age", 18, 100)
    _check_int("creditScore", 100, 1000)
    _check_float("balance", 0.0)
    _check_float("estimatedSalary", 0.0)
    _check_int("tenure", 0, 20)
    _check_int("numOfProducts", 1, 4)
    _check_int("satisfactionScore", 1, 5)

    for b in ["isActiveMember","hasCrCard","complain"]:
        if b in payload and payload[b] is not None:
            if _to_bool01(payload[b]) not in {0,1}:
                w.append(f"{b} debe ser booleano (true/false) o 0/1")
    return w

example_v4 = {
  "geography":"Spain","gender":"Male","age":42,"creditScore":650,"balance":1200.5,
  "estimatedSalary":45000,"tenure":6,"numOfProducts":2,"satisfactionScore":2,
  "isActiveMember":True,"hasCrCard":True,"complain":False
}
print("Warnings:", validate_domain_v4(example_v4))
contract_v4_to_df(example_v4)


## 4) Preparación y pipeline (preprocess + SMOTE + Logistic Regression)

In [ ]:
FEATURES_DS = [
    "Geography","Gender","Age","CreditScore","Balance","EstimatedSalary","Tenure",
    "NumOfProducts","SatisfactionScore","IsActiveMember","HasCrCard","Complain"
]

# Intento mapear si el dataset viene en camelCase
alt_map = {
    "geography":"Geography","gender":"Gender","age":"Age","creditScore":"CreditScore",
    "balance":"Balance","estimatedSalary":"EstimatedSalary","tenure":"Tenure",
    "numOfProducts":"NumOfProducts","satisfactionScore":"SatisfactionScore",
    "isActiveMember":"IsActiveMember","hasCrCard":"HasCrCard","complain":"Complain"
}
for src,dst in alt_map.items():
    if src in df.columns and dst not in df.columns:
        df = df.rename(columns={src:dst})

# Crea faltantes como NaN (imputable)
for c in FEATURES_DS:
    if c not in df.columns:
        df[c] = np.nan

X = df[FEATURES_DS].copy()
y = df["Exited"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

categorical = ["Geography","Gender"]
numeric = [c for c in FEATURES_DS if c not in categorical]

numeric_pipe = ImbPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = ImbPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric),
        ("cat", categorical_pipe, categorical)
    ],
    remainder="drop"
)

smote = SMOTE(random_state=SEED, k_neighbors=5)
clf = LogisticRegression(max_iter=2000, solver="lbfgs")

pipe = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", smote),
    ("model", clf)
])

pipe


## 5) Entrenamiento + evaluación

In [ ]:
pipe.fit(X_train, y_train)

y_proba = pipe.predict_proba(X_test)[:, 1]
y_pred_05 = (y_proba >= 0.5).astype(int)

print("=== Evaluación con threshold=0.50 (referencia) ===")
print(classification_report(y_test, y_pred_05))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_05))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 4))

try:
    RocCurveDisplay.from_predictions(y_test, y_proba)
except Exception as e:
    print("ROC curve no disponible:", e)


## 6) Threshold de negocio (por defecto 0.22)

In [ ]:
y_pred_thr = (y_proba >= THRESHOLD).astype(int)

print(f"=== Evaluación con threshold={THRESHOLD:.2f} (negocio) ===")
print(classification_report(y_test, y_pred_thr))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_thr))


## 7) Explicabilidad (coeficientes) y top_features

In [ ]:
def _get_feature_names(pre: ColumnTransformer):
    num_features = numeric
    ohe = pre.named_transformers_["cat"].named_steps["onehot"]
    cat_features = ohe.get_feature_names_out(categorical).tolist()
    return list(num_features) + cat_features

def global_feature_importance(pipe, top_n=20):
    model = pipe.named_steps["model"]
    feats = _get_feature_names(pipe.named_steps["preprocess"])
    coefs = model.coef_.ravel()
    imp = pd.DataFrame({"feature":feats, "coef":coefs, "abs_coef":np.abs(coefs)})
    return imp.sort_values("abs_coef", ascending=False).head(top_n)

global_feature_importance(pipe, top_n=20)


## 8) predict_v4(payload) — salida para backend

In [ ]:
HUMAN_LABELS = {
    "Complain": "Tiene quejas",
    "IsActiveMember": "Cliente activo",
    "Balance": "Saldo",
    "CreditScore": "Puntaje de crédito",
    "Age": "Edad",
    "NumOfProducts": "Número de productos",
    "SatisfactionScore": "Satisfacción",
    "EstimatedSalary": "Salario estimado",
    "HasCrCard": "Tiene tarjeta",
    "Tenure": "Antigüedad",
    "Geography": "País",
    "Gender": "Género",
}

def _top_features_for_instance(pipe, X_one: pd.DataFrame, top_k=3):
    pre = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]
    feats = _get_feature_names(pre)
    coef = model.coef_.ravel()

    Xt = pre.transform(X_one)
    x_vec = Xt.toarray().ravel() if hasattr(Xt, "toarray") else np.array(Xt).ravel()

    contrib = coef * x_vec
    idx = np.argsort(np.abs(contrib))[::-1][:top_k]

    out = []
    for i in idx:
        f = feats[i]
        c = float(contrib[i])
        impact = "positivo" if c > 0 else "negativo"

        if f.startswith("Geography_"):
            base = "Geography"; val = f.replace("Geography_","")
        elif f.startswith("Gender_"):
            base = "Gender"; val = f.replace("Gender_","")
        else:
            base = f; val = str(X_one.iloc[0].get(base, ""))

        out.append({"name": HUMAN_LABELS.get(base, base), "value": val, "impact": impact})
    return out

def predict_v4(payload: dict, threshold: float = THRESHOLD, top_k: int = 3):
    warns = validate_domain_v4(payload)
    X_one = contract_v4_to_df(payload)
    proba = float(pipe.predict_proba(X_one)[:, 1][0])
    pred = int(proba >= threshold)

    return {
        "forecast": "Va a cancelar" if pred == 1 else "No va a cancelar",
        "probability": round(proba, 4),
        "top_features": _top_features_for_instance(pipe, X_one, top_k=top_k) if pred == 1 else [],
        "warnings": warns
    }

predict_v4(example_v4)


## 9) Serialización (joblib + metadata.json)

In [ ]:
joblib.dump(pipe, MODEL_PATH)

metadata = {
    "model_type": "LogisticRegression + SMOTE",
    "contract_version": "v4",
    "threshold": THRESHOLD,
    "features_ds": FEATURES_DS,
    "backend_fields": CONTRACT_V4_FIELDS,
    "required_fields": sorted(list(V4_REQUIRED)),
    "optional_fields": sorted(list(V4_OPTIONAL)),
    "postprocess_feature_names": _get_feature_names(preprocess),
    "trained_at_utc": datetime.datetime.utcnow().isoformat() + "Z"
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Guardado OK")
print(" -", MODEL_PATH)
print(" -", METADATA_PATH)


## 10) Prueba “pro”: cargar el joblib y volver a predecir

In [ ]:
loaded = joblib.load(MODEL_PATH)
X_one = contract_v4_to_df(example_v4)
proba_loaded = float(loaded.predict_proba(X_one)[:, 1][0])

print("Probability (loaded):", round(proba_loaded, 4))
print("Forecast (loaded):", "Va a cancelar" if proba_loaded >= THRESHOLD else "No va a cancelar")


## 11) Export demo para Power BI (CSV con varios días)

In [ ]:
sample = X_test.sample(n=min(5000, len(X_test)), random_state=SEED).copy()
proba = pipe.predict_proba(sample)[:, 1]
forecast_int = (proba >= THRESHOLD).astype(int)
forecast_label = np.where(forecast_int==1, "Va a cancelar", "No va a cancelar")

out = pd.DataFrame({
    "customerId": np.arange(1, len(sample)+1),
    "probability": np.round(proba, 4),
    "forecast": forecast_int,
    "forecastLabel": forecast_label
})

start = datetime.datetime.now() - datetime.timedelta(days=13)
dates = [start + datetime.timedelta(days=int(i)) for i in np.linspace(0, 13, len(out))]
out["timestamp"] = [d.isoformat(sep=" ", timespec="seconds") for d in dates]
out["date"] = pd.to_datetime(out["timestamp"]).dt.date.astype(str)

out.to_csv(POWERBI_CSV_PATH, index=False, encoding="utf-8")
print("✅ CSV Power BI generado:", POWERBI_CSV_PATH)
out.head()


## 12) Roadmap (siguiente iteración)

- Calibración de probabilidades
- Selección de threshold por costo FN/FP
- SHAP para explicabilidad robusta
- Persistencia de predicciones (DB) + endpoint GET para dashboards vivos
- Monitoreo (drift) + versionado (MLflow)
